### Tool Caling

In [10]:
# tool create
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [138]:

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  Fetch the LIVE exchange rate from the ExchangeRate API.
  Use this tool ONLY when there is no exchange rate provide.
  """
  
  url = f'https://v6.exchangerate-api.com/v6/4edb9626e017d813681eeba8/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def calculate(
    base_currency_value: float,
    conversion_rate: float
) -> float:
    """
    Calculate the converted currency amount using a conversion rate
    already provided by the user. Do not fetch exchange rates.
    """
    return base_currency_value * conversion_rate

In [150]:
get_conversion_factor.invoke({'base_currency':'INR','target_currency':'USD'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1794960001,
 'time_last_update_utc': 'Wed, 18 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1795046401,
 'time_next_update_utc': 'Thu, 19 Nov 2026 00:00:01 +0000',
 'base_code': 'INR',
 'target_code': 'USD',
 'conversion_rate': 0.01069}

In [120]:
calculate.invoke({'base_currency_value':10, 'conversion_rate':93.5163})

935.163

In [140]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [141]:
llm_with_tools = llm.bind_tools([get_conversion_factor, calculate])

In [142]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that convert 10 inr to usd. If you want you can call the all tools at once if needed to get the result caz it has no agent loop')]

In [143]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that convert 10 inr to usd. If you want you can call the all tools at once if needed to get the result caz it has no agent loop', additional_kwargs={}, response_metadata={})]

In [144]:
ai_message = llm_with_tools.invoke(messages)

In [145]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'fc_e0ef6a58-e478-4931-8379-8837cd320bfc',
  'type': 'tool_call'}]

In [146]:
messages1 = [HumanMessage('convert 10 usd to inr . 1 usd = 93.80 inr')]

In [147]:
ai_message1 = llm_with_tools.invoke(messages1)

In [148]:
ai_message1.tool_calls

[{'name': 'calculate',
  'args': {'base_currency_value': 10, 'conversion_rate': 93.8},
  'id': 'fc_3f5c0c5f-e33f-41ae-8cd5-f75ec52f3e8e',
  'type': 'tool_call'}]